<a href="https://colab.research.google.com/github/HitanshuGedam/quantum-learning-journey/blob/main/Day11_Partial_Trace_Reduced_Density_Matrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 11: Partial Trace and Reduced Density Matrices

## Understanding Subsystems in Multi-Qubit Systems

## Learning Objectives

By the end of this notebook, you will understand:

1. What is a partial trace and why we need it
2. How to trace out qubits from a multi-qubit state
3. What reduced density matrices represent physically
4. How partial trace helps detect entanglement
5. The relationship between Schmidt decomposition and partial trace
6. How to compute partial traces using QuTiP

## Prerequisites

Before this notebook, you should understand:
- Density matrices
- Tensor products
- Multi-qubit basis states

## Technical Depth

- Partial trace definition: $\rho_A = \text{Tr}_B(\rho_{AB})$
- Tracing out subsystems
- Entanglement detection via reduced state purity
- Schmidt decomposition and partial trace

## References

- Nielsen & Chuang (2010). Quantum Computation and Quantum Information. Chapter 2.
- QuTiP Documentation: https://qutip.org/
- https://qubit.guide/8.7-partial-trace-revisited

In [1]:
# ============================================================================
# SETUP AND INSTALLATIONS
# ============================================================================

!pip install qutip qutip_qip -q

import numpy as np
import qutip as qt
import matplotlib.pyplot as plt

print(f"NumPy version: {np.__version__}")
print(f"QuTiP version: {qt.__version__}")
print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.8/140.8 kB 5.6 MB/s eta 0:00:00
NumPy version: 2.0.2
QuTiP version: 5.2.3
Setup complete.


## 1. Why Do We Need the Partial Trace?

### The Problem

Imagine you have a two-qubit system, but you only care about what is happening on the first qubit. The second qubit might be:
- In another room
- Measured by someone else
- Just not interesting to you

How do we describe the state of just the first qubit?

### The Solution: Partial Trace

The partial trace is a mathematical operation that "traces out" or removes one subsystem, giving us the reduced density matrix for the remaining subsystem.

### Notation

For a bipartite state $\rho_{AB}$ (two systems A and B), the partial trace over B is written as:

$$ \rho_A = \text{Tr}_B(\rho_{AB}) $$

This gives us the state of system A alone.

### Simple Analogy

Think of a coin toss with two coins:
- You have information about both coins (joint probability distribution)
- But you only care about the first coin
- You "trace out" the second coin by summing over all its possibilities

The partial trace does the same thing for quantum states!

## 2. Definition of Partial Trace

### Mathematical Definition

For a bipartite density matrix $\rho_{AB}$, the partial trace over subsystem B is defined as:

$$ \rho_A = \text{Tr}_B(\rho_{AB}) = \sum_i (I_A \otimes \langle i|_B) \rho_{AB} (I_A \otimes |i\rangle_B) $$

where $\{|i\rangle_B\}$ is an orthonormal basis for subsystem B.

### In Simple Terms

To trace out qubit B:
1. Keep qubit A as is (apply identity $I_A$)
2. Project qubit B onto basis state $|i\rangle$
3. Sum over all basis states $i$

### For a Product State

If $\rho_{AB} = \rho_A \otimes \rho_B$, then:

$$ \text{Tr}_B(\rho_{AB}) = \rho_A \cdot \text{Tr}(\rho_B) = \rho_A $$

Because $\text{Tr}(\rho_B) = 1$ for any density matrix.

### Key Property

The partial trace is the **unique** operation that preserves all measurement statistics on the remaining subsystem.

In [2]:
# ============================================================================
# PARTIAL TRACE FOR PRODUCT STATES
# ============================================================================

print("=" * 70)
print("PARTIAL TRACE FOR PRODUCT STATES")
print("=" * 70)

# Create single-qubit states
ket0 = qt.basis(2, 0)
ket1 = qt.basis(2, 1)

# Create density matrices for each qubit
rho_A = ket0 * ket0.dag()      # |0⟩⟨0|
rho_B = ket1 * ket1.dag()      # |1⟩⟨1|

print("Individual states:")
print(f"ρ_A (qubit A) = {rho_A}")
print(f"ρ_B (qubit B) = {rho_B}")

# Create product state
rho_AB = qt.tensor(rho_A, rho_B)
print(f"\nProduct state ρ_AB = ρ_A ⊗ ρ_B =")
print(rho_AB)

# Compute partial trace over B
rho_A_recovered = rho_AB.ptrace(0)
print(f"\nAfter partial trace over B (keep qubit A):")
print(f"ρ_A = Tr_B(ρ_AB) = {rho_A_recovered}")

# Verify we recovered the original state
is_same = np.allclose(rho_A.full(), rho_A_recovered.full())
print(f"\nRecovered original ρ_A? {is_same}")

print("\n✅ For product states, partial trace recovers the individual state!")

PARTIAL TRACE FOR PRODUCT STATES
Individual states:
ρ_A (qubit A) = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[1. 0.]
 [0. 0.]]
ρ_B (qubit B) = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0.]
 [0. 1.]]

Product state ρ_AB = ρ_A ⊗ ρ_B =
Quantum object: dims=[[2, 2], [2, 2]], shape=(4, 4), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

After partial trace over B (keep qubit A):
ρ_A = Tr_B(ρ_AB) = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[1. 0.]
 [0. 0.]]

Recovered original ρ_A? True

✅ For product states, partial trace recovers the individual state!


## 3. Partial Trace for General States

### The General Formula

For a general 2-qubit density matrix:

$$ \rho_{AB} = \sum_{i,j,k,l} \rho_{ij,kl} |i\rangle\langle j| \otimes |k\rangle\langle l| $$

The partial trace over qubit B is:

$$ \rho_A = \sum_{i,j,k} \rho_{ij,kk} |i\rangle\langle j| $$

In words: We sum over the diagonal elements of the B subsystem.

### Matrix Form

For a 4x4 density matrix written in the basis $|00\rangle, |01\rangle, |10\rangle, |11\rangle$:

$$ \rho_{AB} = \begin{pmatrix}
\rho_{00,00} & \rho_{00,01} & \rho_{00,10} & \rho_{00,11} \\
\rho_{01,00} & \rho_{01,01} & \rho_{01,10} & \rho_{01,11} \\
\rho_{10,00} & \rho_{10,01} & \rho_{10,10} & \rho_{10,11} \\
\rho_{11,00} & \rho_{11,01} & \rho_{11,10} & \rho_{11,11}
\end{pmatrix} $$

The reduced density matrix for qubit A is:

$$ \rho_A = \begin{pmatrix}
\rho_{00,00} + \rho_{01,01} & \rho_{00,10} + \rho_{01,11} \\
\rho_{10,00} + \rho_{11,01} & \rho_{10,10} + \rho_{11,11}
\end{pmatrix} $$

### In Words

We take 2x2 blocks and sum their diagonals.

In [3]:
# ============================================================================
# PARTIAL TRACE FOR GENERAL STATES
# ============================================================================

print("=" * 70)
print("PARTIAL TRACE FOR GENERAL STATES")
print("=" * 70)

# Create a general 2-qubit state
# |ψ⟩ = 0.6|00⟩ + 0.3|01⟩ + 0.2|10⟩ + 0.7|11⟩ (not normalized)
# Let's normalize it
coeffs = [0.6, 0.3, 0.2, 0.7]
norm = np.sqrt(sum(c**2 for c in coeffs))
coeffs = [c / norm for c in coeffs]

print(f"Normalized coefficients: {coeffs}")

# Create the state vector
ket0 = qt.basis(2, 0)
ket1 = qt.basis(2, 1)

ket00 = qt.tensor(ket0, ket0)
ket01 = qt.tensor(ket0, ket1)
ket10 = qt.tensor(ket1, ket0)
ket11 = qt.tensor(ket1, ket1)

psi = coeffs[0] * ket00 + coeffs[1] * ket01 + coeffs[2] * ket10 + coeffs[3] * ket11
psi = psi.unit()

print(f"\nState |ψ⟩ = {psi}")

# Create density matrix
rho_AB = psi * psi.dag()

print(f"\nDensity matrix ρ_AB = |ψ⟩⟨ψ|:")
print(rho_AB)

# Compute partial trace using QuTiP
rho_A = rho_AB.ptrace(0)  # Keep qubit A, trace out qubit B
rho_B = rho_AB.ptrace(1)  # Keep qubit B, trace out qubit A

print(f"\nReduced density matrix for qubit A:")
print(rho_A)

print(f"\nReduced density matrix for qubit B:")
print(rho_B)

# Verify traces
print(f"\nTr(ρ_AB) = {rho_AB.tr():.3f}")
print(f"Tr(ρ_A) = {rho_A.tr():.3f}")
print(f"Tr(ρ_B) = {rho_B.tr():.3f}")

print("\n✅ Partial trace preserves the total trace!")

PARTIAL TRACE FOR GENERAL STATES
Normalized coefficients: [np.float64(0.6060915267313264), np.float64(0.3030457633656632), np.float64(0.20203050891044216), np.float64(0.7071067811865475)]

State |ψ⟩ = Quantum object: dims=[[2, 2], [1]], shape=(4, 1), type='ket', dtype=Dense
Qobj data =
[[0.60609153]
 [0.30304576]
 [0.20203051]
 [0.70710678]]

Density matrix ρ_AB = |ψ⟩⟨ψ|:
Quantum object: dims=[[2, 2], [2, 2]], shape=(4, 4), type='oper', dtype=Dense, isherm=True
Qobj data =
[[0.36734694 0.18367347 0.12244898 0.42857143]
 [0.18367347 0.09183673 0.06122449 0.21428571]
 [0.12244898 0.06122449 0.04081633 0.14285714]
 [0.42857143 0.21428571 0.14285714 0.5       ]]

Reduced density matrix for qubit A:
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
Qobj data =
[[0.45918367 0.33673469]
 [0.33673469 0.54081633]]

Reduced density matrix for qubit B:
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
Qobj data =
[[0.40816327 0

## 4. Physical Meaning of Reduced Density Matrices

### What Does ρ_A Represent?

The reduced density matrix $\rho_A$ contains all the information we can obtain by measuring only subsystem A.

### Key Properties

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Property</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Meaning</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">$\text{Tr}(\rho_A) = 1$</td>
            <td style="padding: 8px;">Probabilities sum to 1</td>
        </tr>
        <tr>
            <td style="padding: 8px;">$\rho_A$ is Hermitian</td>
            <td style="padding: 8px;">Eigenvalues are real</td>
        </tr>
        <tr>
            <td style="padding: 8px;">$\rho_A$ is positive</td>
            <td style="padding: 8px;">All eigenvalues >= 0</td>
        </tr>
        <tr>
            <td style="padding: 8px;">$p(m) = \text{Tr}(P_m \rho_A)$</td>
            <td style="padding: 8px;">Born rule for measurements on A</td>
        </tr>
    </tbody>
</table>

### Example: Measuring Only the First Qubit

If we have a two-qubit state and we only measure the first qubit, the probabilities are given by $\rho_A$.

The probability of getting $|0\rangle$ on the first qubit is:

$$ p(0) = \langle 0| \rho_A |0\rangle $$

This equals the sum of probabilities of $|00\rangle$ and $|01\rangle$ in the original state.

In [6]:
# ============================================================================
# PHYSICAL INTERPRETATION OF REDUCED DENSITY MATRICES
# ============================================================================

print("=" * 70)
print("PHYSICAL INTERPRETATION")
print("=" * 70)

# Create a Bell state
ket0 = qt.basis(2, 0)
ket1 = qt.basis(2, 1)

bell_state = (qt.tensor(ket0, ket0) + qt.tensor(ket1, ket1)).unit()
rho_bell = bell_state * bell_state.dag()

print("Bell state |Φ⁺⟩ = (|00⟩ + |11⟩)/√2")
print(f"rho_bell = {rho_bell}")

# Compute reduced density matrix for first qubit
rho_A = rho_bell.ptrace(0)
print(f"\nReduced density matrix for first qubit:")
print(rho_A)

# Probability of measuring |0⟩ on first qubit
prob_0 = (qt.basis(2, 0).dag() * rho_A * qt.basis(2, 0)).real
prob_1 = (qt.basis(2, 1).dag() * rho_A * qt.basis(2, 1)).real

print(f"\nProbability of measuring |0⟩ on first qubit: {prob_0:.3f}")
print(f"Probability of measuring |1⟩ on first qubit: {prob_1:.3f}")

# Calculate probabilities directly using amplitudes
# The Bell state has coefficients: 1/√2 for |00⟩ and 1/√2 for |11⟩
amplitude_00 = 1 / np.sqrt(2)
amplitude_01 = 0
amplitude_10 = 0
amplitude_11 = 1 / np.sqrt(2)

prob_00 = abs(amplitude_00)**2
prob_01 = abs(amplitude_01)**2
prob_10 = abs(amplitude_10)**2
prob_11 = abs(amplitude_11)**2

print(f"\nDirect calculation from Bell state coefficients:")
print(f"  P(|00⟩) = |1/√2|² = {prob_00:.3f}")
print(f"  P(|01⟩) = {prob_01:.3f}")
print(f"  P(|10⟩) = {prob_10:.3f}")
print(f"  P(|11⟩) = |1/√2|² = {prob_11:.3f}")
print(f"  So P(first qubit = 0) = P(|00⟩) + P(|01⟩) = {prob_00 + prob_01:.3f}")
print(f"  So P(first qubit = 1) = P(|10⟩) + P(|11⟩) = {prob_10 + prob_11:.3f}")

print("\n✅ ρ_A correctly predicts measurement probabilities on subsystem A!")

PHYSICAL INTERPRETATION
Bell state |Φ⁺⟩ = (|00⟩ + |11⟩)/√2
rho_bell = Quantum object: dims=[[2, 2], [2, 2]], shape=(4, 4), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.5 0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0.5]]

Reduced density matrix for first qubit:
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.5 0. ]
 [0.  0.5]]

Probability of measuring |0⟩ on first qubit: 0.500
Probability of measuring |1⟩ on first qubit: 0.500

Direct calculation from Bell state coefficients:
  P(|00⟩) = |1/√2|² = 0.500
  P(|01⟩) = 0.000
  P(|10⟩) = 0.000
  P(|11⟩) = |1/√2|² = 0.500
  So P(first qubit = 0) = P(|00⟩) + P(|01⟩) = 0.500
  So P(first qubit = 1) = P(|10⟩) + P(|11⟩) = 0.500

✅ ρ_A correctly predicts measurement probabilities on subsystem A!


## 5. Partial Trace and Entanglement Detection

### Key Insight

For a pure bipartite state $|\psi_{AB}\rangle$:
- If the state is **separable** (product state), then $\rho_A$ is a pure state
- If the state is **entangled**, then $\rho_A$ is a mixed state

### Why?

A separable state can be written as $|\psi_A\rangle \otimes |\psi_B\rangle$. Then:

$$ \rho_A = \text{Tr}_B(|\psi_A\rangle\langle\psi_A| \otimes |\psi_B\rangle\langle\psi_B|) = |\psi_A\rangle\langle\psi_A| $$

This is a pure state (purity = 1).

For an entangled state, the reduced state becomes mixed (purity < 1).

### Purity as Entanglement Witness

The purity of $\rho_A$ tells us about entanglement:

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Purity of $\rho_A$</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Conclusion</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">$1$</td>
            <td style="padding: 8px;">Global state is separable (product state)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">$< 1$</td>
            <td style="padding: 8px;">Global state is entangled</td>
        </tr>
        <tr>
            <td style="padding: 8px;">$0.5$ (for qubits)</td>
            <td style="padding: 8px;">Maximally entangled</td>
        </tr>
    </tbody>
</table>

In [8]:
# ============================================================================
# USING PARTIAL TRACE TO DETECT ENTANGLEMENT
# ============================================================================

print("=" * 70)
print("DETECTING ENTANGLEMENT USING PARTIAL TRACE")
print("=" * 70)

def purity(rho):
    """Calculate purity Tr(ρ²) of a density matrix."""
    return (rho * rho).tr().real

# Create basis states
ket0 = qt.basis(2, 0)
ket1 = qt.basis(2, 1)
ket_plus = (ket0 + ket1).unit()

# ============================================================================
# 1. Product state: |0⟩|0⟩
# ============================================================================
product_ket = qt.tensor(ket0, ket0)
rho_product = product_ket * product_ket.dag()

# ============================================================================
# 2. Product state: |+⟩|0⟩
# ============================================================================
product2_ket = qt.tensor(ket_plus, ket0)
rho_product2 = product2_ket * product2_ket.dag()

# ============================================================================
# 3. Partially entangled state: cosθ|00⟩ + sinθ|11⟩ with θ = 30°
# ============================================================================
theta = np.pi/6  # 30 degrees
cos_t = np.cos(theta)
sin_t = np.sin(theta)
partial_ket = cos_t * qt.tensor(ket0, ket0) + sin_t * qt.tensor(ket1, ket1)
partial_ket = partial_ket.unit()
rho_partial = partial_ket * partial_ket.dag()

# ============================================================================
# 4. Maximally entangled state: Bell state |Φ⁺⟩
# ============================================================================
bell_ket = (qt.tensor(ket0, ket0) + qt.tensor(ket1, ket1)).unit()
rho_bell = bell_ket * bell_ket.dag()

states = [
    ("Product: |0⟩|0⟩", rho_product),
    ("Product: |+⟩|0⟩", rho_product2),
    ("Partially entangled (θ=30°)", rho_partial),
    ("Maximally entangled (Bell)", rho_bell)
]

print("\n" + "=" * 60)
print("Detecting entanglement via reduced state purity")
print("=" * 60)

for name, rho in states:
    # Compute reduced state for qubit A
    rho_A = rho.ptrace(0)
    pur = purity(rho_A)

    # Get eigenvalues of reduced state
    evals = rho_A.eigenenergies()
    # Filter out tiny numerical zeros
    evals = evals[evals > 1e-10]

    # Determine if entangled
    if abs(pur - 1.0) < 1e-6:
        status = "SEPARABLE (product state)"
    elif pur < 0.51 and pur > 0.49:
        status = "MAXIMALLY ENTANGLED"
    else:
        status = "ENTANGLED (partial)"

    print(f"\n{name}:")
    print(f"  ρ_A = {rho_A}")

    # Print eigenvalues safely
    if len(evals) == 1:
        print(f"  Eigenvalues of ρ_A: {evals[0]:.4f}")
    else:
        print(f"  Eigenvalues of ρ_A: {evals[0]:.4f}, {evals[1]:.4f}")

    print(f"  Purity of ρ_A = {pur:.4f}")
    print(f"  Status: {status}")

print("\n" + "=" * 60)
print("KEY INSIGHT")
print("=" * 60)
print("""
- Separable (product) states: Reduced state is PURE (purity = 1)
- Entangled states: Reduced state is MIXED (purity < 1)
- Maximally entangled: Reduced state is MAXIMALLY MIXED (purity = 0.5)

The partial trace reveals entanglement through the mixedness of subsystems!
""")
print("=" * 60)

DETECTING ENTANGLEMENT USING PARTIAL TRACE

Detecting entanglement via reduced state purity

Product: |0⟩|0⟩:
  ρ_A = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[1. 0.]
 [0. 0.]]
  Eigenvalues of ρ_A: 1.0000
  Purity of ρ_A = 1.0000
  Status: SEPARABLE (product state)

Product: |+⟩|0⟩:
  ρ_A = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.5 0.5]
 [0.5 0.5]]
  Eigenvalues of ρ_A: 1.0000
  Purity of ρ_A = 1.0000
  Status: SEPARABLE (product state)

Partially entangled (θ=30°):
  ρ_A = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.75 0.  ]
 [0.   0.25]]
  Eigenvalues of ρ_A: 0.2500, 0.7500
  Purity of ρ_A = 0.6250
  Status: ENTANGLED (partial)

Maximally entangled (Bell):
  ρ_A = Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0.5 0. ]
 [0.  0.5]]
  Eigenvalues of ρ_A: 0.5000, 0.5000


## 6. Schmidt Decomposition and Partial Trace

### The Schmidt Decomposition

Any pure bipartite state can be written as:

$$ |\psi_{AB}\rangle = \sum_{i=1}^{r} \lambda_i |i_A\rangle |i_B\rangle $$

where $\lambda_i > 0$ are Schmidt coefficients and $\sum_i \lambda_i^2 = 1$.

### Connection to Partial Trace

If we have the Schmidt decomposition, the reduced density matrices are:

$$ \rho_A = \sum_i \lambda_i^2 |i_A\rangle\langle i_A| $$

$$ \rho_B = \sum_i \lambda_i^2 |i_B\rangle\langle i_B| $$

### Key Insight

The Schmidt coefficients appear directly as the eigenvalues of $\rho_A$ (and $\rho_B$)!

- $\rho_A$ and $\rho_B$ have the same non-zero eigenvalues
- These eigenvalues are $\lambda_i^2$
- The number of non-zero eigenvalues = Schmidt rank

### This tells us:

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Schmidt rank</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Eigenvalues of $\rho_A$</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">State type</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">1</td>
            <td style="padding: 8px;">$[1]$</td>
            <td style="padding: 8px;">Separable (product)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">2</td>
            <td style="padding: 8px;">$[\lambda_1^2, \lambda_2^2]$</td>
            <td style="padding: 8px;">Entangled</td>
        </tr>
        <tr>
            <td style="padding: 8px;">2</td>
            <td style="padding: 8px;">$[0.5, 0.5]$</td>
            <td style="padding: 8px;">Maximally entangled</td>
        </tr>
    </tbody>
</table>

## 7. Summary and Key Insights

### Partial Trace Summary

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Concept</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Formula</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Partial trace over B</td>
            <td style="padding: 8px;">$\rho_A = \text{Tr}_B(\rho_{AB})$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">For product state</td>
            <td style="padding: 8px;">$\text{Tr}_B(\rho_A \otimes \rho_B) = \rho_A$</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Reduced state eigenvalues</td>
            <td style="padding: 8px;">$\lambda_i^2$ (Schmidt coefficients squared)</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Purity of $\rho_A$</td>
            <td style="padding: 8px;">$\text{Tr}(\rho_A^2) = \sum_i \lambda_i^4$</td>
        </tr>
    </tbody>
</table>

### Entanglement Detection via Partial Trace

<table style="width: 100%; border-collapse: collapse;">
    <thead>
        <tr>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">State type</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Global state purity</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Reduced state purity</th>
            <th style="text-align: left; padding: 8px; background-color: #f2f2f2;">Entangled?</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td style="padding: 8px;">Product (separable)</td>
            <td style="padding: 8px;">1</td>
            <td style="padding: 8px;">1</td>
            <td style="padding: 8px;">No</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Partially entangled</td>
            <td style="padding: 8px;">1</td>
            <td style="padding: 8px;">between 0.5 and 1</td>
            <td style="padding: 8px;">Yes</td>
        </tr>
        <tr>
            <td style="padding: 8px;">Maximally entangled</td>
            <td style="padding: 8px;">1</td>
            <td style="padding: 8px;">0.5</td>
            <td style="padding: 8px;">Yes (maximum)</td>
        </tr>
    </tbody>
</table>

### Key Formulas (LaTeX)

$$ \rho_A = \text{Tr}_B(\rho_{AB}) = \sum_i (I_A \otimes \langle i|_B) \rho_{AB} (I_A \otimes |i\rangle_B) $$

$$ |\psi_{AB}\rangle = \sum_i \lambda_i |i_A\rangle |i_B\rangle $$

$$ \rho_A = \sum_i \lambda_i^2 |i_A\rangle\langle i_A| $$

$$ \text{Tr}(\rho_A^2) = \sum_i \lambda_i^4 $$


### Key Takeaways

1. **Partial trace** removes (traces out) a subsystem, giving the state of the remaining part
2. **Reduced density matrices** contain all measurable information about a subsystem
3. **For product states:** $\text{Tr}_B(\rho_A \otimes \rho_B) = \rho_A$ (pure reduced state)
4. **For entangled states:** Reduced states are mixed, revealing entanglement
5. **Schmidt decomposition** connects directly to partial trace: eigenvalues of $\rho_A$ are $\lambda_i^2$
6. **QuTiP's `ptrace()`** method computes partial trace easily

---

**Day 11 Complete!** 🎉

You now understand:
- What is a partial trace and why we need it
- How to trace out qubits from a multi-qubit state
- What reduced density matrices represent physically
- How partial trace helps detect entanglement
- The relationship between Schmidt decomposition and partial trace
- How to compute partial traces using QuTiP

Proceed to Day 12 when ready.